# 01b_primer_equipo
Amplía el universo de jugadores con los que **empezaron su carrera en uno de los clubes** (o en un juvenil de uno de ellos) pero nunca figuraron en una página juvenil (carreras viejas, sin detalle de inferiores).

**Lógica:**
1. Por cada club de `bronze_clubs` y cada temporada 1985-hoy (en la división que sea) se baja la página de transfers y se guardan **solo las entradas**.
2. Los jugadores de esas entradas que no están en el universo (`bronze_arrivals` + `bronze_departures` + ya incorporados + ya decididos) son candidatos.
3. A cada candidato se le hace **solo la llamada `transfers`**. Su primer club (el origen del traspaso más viejo; si es un marcador tipo Unknown/Without Club, el destino) decide:
   - club padre de la lista -> `aceptado` / `primer_club_padre`
   - juvenil de un padre -> `aceptado` / `primer_club_juvenil_de_padre` **(diagnóstico: el proceso actual no lo captó; mirar por qué)**
   - cualquier otro club -> `descartado` (queda anotado para no volver a llamarlo)
   - un aceptado cuyo primer registro es anterior a `FECHA_CORTE` -> `descartado` / `primer_registro_pre_corte`
4. Los aceptados entran a `bronze_universo_extra` y su respuesta de `transfers` se guarda en `bronze_player_raw` (así `02_jugadores` no la repite y baja las otras 5).

**Tablas que escribe:** `bronze_pe_arrivals`, `bronze_pe_status`, `bronze_candidatos`, `bronze_universo_extra` (y filas en `bronze_player_raw` y `bronze_corridas`).
Los descartados NUNCA se escriben en `bronze_player_raw` (silver lee todo lo que hay ahí).

**Se puede cortar y retomar.** Si queda algo pendiente (tope de horas, freno), termina en rojo para que `02_jugadores` espere.
Parámetros de prueba: `LIMITE_PAGINAS` y `LIMITE_JUGADORES` (0 = sin límite).

## Contenido

| Celda | Qué hace |
|---|---|
| `%run ./ciclo` | Carga el estado del ciclo semanal |
| 1) Configuración y funciones | Widgets, tablas, descarga con caché y llamadas a la API |
| 2) Compuerta del ciclo y pausa por freno | Sale si no corresponde correr o si hay una pausa vigente |
| 3) Fase A | Baja las entradas de cada club en cada temporada |
| 4) Fase B | Decide qué candidatos entran al universo |
| 5) Resumen y cierre | Consultas de control y salida del notebook |

## Widgets

| Widget | Valor por defecto | Para qué sirve |
|---|---|---|
| `ESPERA_API` | `1.0` | Segundos de espera antes de cada llamada a la API |
| `MAX_HORAS` | `7` | Tope de horas de la corrida (se mide desde el inicio del notebook) |
| `MAX_INTENTOS` | `3` | Errores por jugador dentro del ciclo antes de dejar de reintentarlo |
| `MAX_ERRORES_SEGUIDOS` | `5` | Errores consecutivos de la API que cortan la corrida |
| `VOLCAR_CADA` | `20` | Cada cuántas decisiones de candidatos se escribe en las tablas |
| `PAUSA_HORAS` | `6` | Duración de la pausa que se registra ante un freno del sitio |
| `API_REF` | commit de `felipeall/transfermarkt-api` | Versión de la API que se descarga |
| `LIMITE_PAGINAS` | `0` | Máximo de páginas de la fase A (0 = sin límite) |
| `LIMITE_JUGADORES` | `0` | Máximo de candidatos de la fase B (0 = sin límite) |
| `FECHA_CORTE` | `1985-07-01` | Inicio de la temporada 85/86, la primera del alcance |

## Tablas que lee

`bronze_clubs`, `bronze_youth_teams_v2`, `bronze_clubes_externos` (si existe), `bronze_arrivals`, `bronze_departures`, `bronze_ciclos` (a través de `ciclo`) y `bronze_corridas` (para la pausa vigente).

## Reglas de acceso

- Páginas de `www.transfermarkt.com`: consulta `robots.txt` antes de cada pedido no cacheado, espera 3 s (`DELAY`) y corta ante HTTP `202`, `403` o `429`.
- API local: espera `ESPERA_API` s antes de cada llamada. Ante `202`, `403` o `429` corta la corrida y registra una pausa de `PAUSA_HORAS` horas en `bronze_corridas`; mientras la pausa esté vigente, el notebook sale sin hacer pedidos.

## Carga de `ciclo`
`%run ./ciclo` define las funciones del control del ciclo (ver el notebook `ciclo`). Si no se ejecuta, el notebook corre en **modo prueba**: sin compuerta ni ciclo, y "este ciclo" pasa a ser esta corrida.

In [ ]:
%run ./ciclo

## 1) Configuración y funciones
Crea los widgets, define los nombres de las tablas, los esquemas y las tablas Delta que no existan, y las funciones auxiliares.

| Elemento | Qué hace |
|---|---|
| `MODO_PRUEBA` | `True` si `ciclo` no se cargó (no existe la función `ahora`) |
| `ANIO`, `TEMPORADAS`, `DESDE` | Año en curso, temporadas de 1985 a `ANIO` y año desde el cual las páginas se vuelven a bajar una vez por ciclo (`ANIO - 1`) |
| `Freno` | Excepción que lanza `get` ante HTTP `202`, `403` o `429` |
| `Corte(msg, fallo, pausa)` | Excepción que corta una fase. `fallo` hace que el notebook termine en error; `pausa` registra una pausa |
| `registrar(motivo, ...)` | Agrega una fila a `bronze_corridas` con el motivo prefijado por `primer_equipo: ` |
| `curl(url, out)` y `get(url, fresco)` | Descarga con caché en el Volume, `robots.txt`, espera y reintentos. Con `fresco=True` borra antes la copia en caché de esa URL |
| `partir(page)` | Devuelve solo el bloque de entradas (entre `<h2>Arrivals</h2>` y `<h2>Departures</h2>`) |
| `filas(bloque)` | Extrae de cada fila: id y nombre del jugador, edad, id y nombre del otro club, `transfer_id` y `fee` |
| `procesar(team_id, slug, y)` | Baja la página de transfers y devuelve el estado y las filas |
| `levantar_api()` | Si la API no responde en `127.0.0.1:8000`, descarga el código de `felipeall/transfermarkt-api` en el commit `API_REF`, lo instala, lo arranca con `uvicorn` y espera hasta 60 s |
| `llamar(pid, ll)` | Hace `GET /players/{pid}/{ll}` y devuelve `(código HTTP, cuerpo)`. Devuelve `-1` ante un error de conexión y `-2` ante cualquier otro error |

**Estados que devuelve `procesar`**

| Estado | Significado |
|---|---|
| `ok` | La página pasó las validaciones |
| `sin_pagina` | HTTP 404 |
| `http_<código>` | Otro código distinto de 200 |
| `titulo_distinto` | El título no trae `Transfers YY/YY` de esa temporada |
| `sin_h2` | Falta `<h2>Arrivals</h2>` o `<h2>Departures</h2>` |

In [ ]:
# 1) Configuracion y funciones
import json, os, re, sys, time, hashlib, subprocess, urllib.request, urllib.error, urllib.robotparser, zipfile
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

dbutils.widgets.text("ESPERA_API", "1.0")
dbutils.widgets.text("MAX_HORAS", "7")
dbutils.widgets.text("MAX_INTENTOS", "3")
dbutils.widgets.text("MAX_ERRORES_SEGUIDOS", "5")
dbutils.widgets.text("VOLCAR_CADA", "20")
dbutils.widgets.text("PAUSA_HORAS", "6")
dbutils.widgets.text("API_REF", "71aa75ecd708e5c8c283c4af5968dd370777b3f8")
dbutils.widgets.text("LIMITE_PAGINAS", "0")
dbutils.widgets.text("LIMITE_JUGADORES", "0")
dbutils.widgets.text("FECHA_CORTE", "1985-07-01")   # inicio de la temporada 85/86, la primera del alcance

ESPERA_API = float(dbutils.widgets.get("ESPERA_API"))
MAX_HORAS = float(dbutils.widgets.get("MAX_HORAS"))
MAX_INTENTOS = int(dbutils.widgets.get("MAX_INTENTOS"))
MAX_ERRORES_SEGUIDOS = int(dbutils.widgets.get("MAX_ERRORES_SEGUIDOS"))
VOLCAR_CADA = int(dbutils.widgets.get("VOLCAR_CADA"))
PAUSA_HORAS = float(dbutils.widgets.get("PAUSA_HORAS"))
API_REF = dbutils.widgets.get("API_REF")
LIMITE_PAGINAS = int(dbutils.widgets.get("LIMITE_PAGINAS"))
LIMITE_JUGADORES = int(dbutils.widgets.get("LIMITE_JUGADORES"))
FECHA_CORTE = dbutils.widgets.get("FECHA_CORTE")
ANIO_CORTE = int(FECHA_CORTE[:4])

SCH = "workspace.inferiores"
T_CLUBS = f"{SCH}.bronze_clubs"
T_YOUTH = f"{SCH}.bronze_youth_teams_v2"
T_EXT = f"{SCH}.bronze_clubes_externos"
T_ARR = f"{SCH}.bronze_arrivals"
T_DEP = f"{SCH}.bronze_departures"
T_PE_ARR = f"{SCH}.bronze_pe_arrivals"
T_PE_EST = f"{SCH}.bronze_pe_status"
T_CAND = f"{SCH}.bronze_candidatos"
T_EXTRA = f"{SCH}.bronze_universo_extra"
RAW = f"{SCH}.bronze_player_raw"
CORR = f"{SCH}.bronze_corridas"

TM = "https://www.transfermarkt.com"
UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
CACHE = "/Volumes/workspace/inferiores/scraper/cache"
DELAY = 3.0
API = "http://127.0.0.1:8000"

ANIO = datetime.now().year
TEMPORADAS = list(range(1985, ANIO + 1))
DESDE = ANIO - 1          # las temporadas >= DESDE se vuelven a bajar una vez por ciclo
LOTE = 50

# Modo prueba: si no se corrio `%run ./ciclo`, el notebook funciona solo (sin compuerta ni ciclo).
try:
    ahora
    MODO_PRUEBA = False
except NameError:
    MODO_PRUEBA = True

    def ahora():
        return datetime.now().isoformat(timespec="seconds")

INICIO = ahora()
T0 = time.time()

SCHEMA_RAW = StructType([
    StructField("player_id", StringType()), StructField("llamada", StringType()),
    StructField("estado", StringType()), StructField("http", IntegerType()),
    StructField("body", StringType()), StructField("fecha", StringType()),
])
SCHEMA_CORR = StructType([
    StructField("inicio", StringType()), StructField("fin", StringType()),
    StructField("llamadas", IntegerType()), StructField("errores", IntegerType()),
    StructField("motivo", StringType()), StructField("pausa_hasta", StringType()),
])
ESQ_LLEG = StructType([StructField(c, t) for c, t in [
    ("team_id", StringType()), ("team_slug", StringType()), ("season", IntegerType()),
    ("player_id", StringType()), ("player_name", StringType()), ("age", StringType()),
    ("counterpart_id", StringType()), ("counterpart_name", StringType()),
    ("transfer_id", StringType()), ("fee", StringType())]])
ESQ_EST = StructType([StructField(c, t) for c, t in [
    ("team_id", StringType()), ("season", IntegerType()), ("estado", StringType()),
    ("n_arrivals", IntegerType()), ("bajado_en", StringType())]])
ESQ_CAND = StructType([StructField(c, t) for c, t in [
    ("player_id", StringType()), ("decision", StringType()), ("motivo", StringType()),
    ("club_primero_id", StringType()), ("club_primero_nombre", StringType()),
    ("edad_primer_registro", IntegerType()), ("fecha_primer_registro", StringType()),
    ("n_traspasos", IntegerType()), ("fecha_decision", StringType())]])
ESQ_EXTRA = StructType([StructField(c, t) for c, t in [
    ("player_id", StringType()), ("season", IntegerType()), ("club_primero_id", StringType()),
    ("motivo", StringType()), ("edad_primer_registro", IntegerType()),
    ("fecha_primer_registro", StringType()), ("incorporado_en", StringType())]])

spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_PE_ARR}
  (team_id STRING, team_slug STRING, season INT, player_id STRING, player_name STRING, age STRING,
   counterpart_id STRING, counterpart_name STRING, transfer_id STRING, fee STRING) USING DELTA""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_PE_EST}
  (team_id STRING, season INT, estado STRING, n_arrivals INT, bajado_en STRING) USING DELTA""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_CAND}
  (player_id STRING, decision STRING, motivo STRING, club_primero_id STRING, club_primero_nombre STRING,
   edad_primer_registro INT, fecha_primer_registro STRING, n_traspasos INT, fecha_decision STRING) USING DELTA""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {T_EXTRA}
  (player_id STRING, season INT, club_primero_id STRING, motivo STRING, edad_primer_registro INT,
   fecha_primer_registro STRING, incorporado_en STRING) USING DELTA""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {RAW}
  (player_id STRING, llamada STRING, estado STRING, http INT, body STRING, fecha STRING) USING DELTA""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {CORR}
  (inicio STRING, fin STRING, llamadas INT, errores INT, motivo STRING, pausa_hasta STRING) USING DELTA""")


class Freno(Exception):
    pass


class Corte(Exception):
    def __init__(self, msg, fallo=False, pausa=False):
        super().__init__(msg)
        self.fallo = fallo
        self.pausa = pausa


def registrar(motivo, llamadas=0, errores=0, pausa=None):
    spark.createDataFrame([(INICIO, ahora(), llamadas, errores, "primer_equipo: " + motivo, pausa)], SCHEMA_CORR) \
         .write.mode("append").saveAsTable(CORR)


def curl(url, out):
    r = subprocess.run(["curl", "-sL", "--max-time", "40", "-A", UA, "-w", "%{http_code}", "-o", out, url],
                       capture_output=True, text=True, timeout=60)
    return r.stdout.strip()


curl(TM + "/robots.txt", "/tmp/robots.txt")
rp = urllib.robotparser.RobotFileParser()
rp.parse(open("/tmp/robots.txt", encoding="utf-8", errors="replace").read().splitlines())


def get(url, fresco=False):
    # `fresco=True` borra el cache de esa URL antes de pedirla (las temporadas recientes cambian).
    p = f"{CACHE}/{hashlib.md5(url.encode()).hexdigest()}.html"
    if fresco and os.path.exists(p):
        os.remove(p)
    if os.path.exists(p):
        return "200", open(p, encoding="utf-8", errors="replace").read()
    if not rp.can_fetch(UA, url):
        raise RuntimeError(f"robots.txt prohibe: {url}")
    time.sleep(DELAY)
    tmp = p + ".tmp"
    code = ""
    for attempt in range(3):
        code = curl(url, tmp)
        if code == "200":
            os.replace(tmp, p)
            return code, open(p, encoding="utf-8", errors="replace").read()
        if code in ("202", "403", "429"):
            raise Freno(f"HTTP {code} en {url}")
        if code == "404":
            return code, ""
        time.sleep(5 * (attempt + 1))
    return code or "0", ""


# --- parser de la pagina de transfers (copiado de la fase 3 de 01_extraccion) ---
def partir(page):
    a = re.search(r"<h2[^>]*>\s*Arrivals\s*</h2>", page)
    d = re.search(r"<h2[^>]*>\s*Departures\s*</h2>", page)
    return page[a.start():d.start()]


def filas(bloque):
    out = []
    patron = r'<tr class="(?:odd|even)">(.*?)</tr>\s*(?=<tr class="(?:odd|even)">|</tbody>)'
    for tr in re.findall(patron, bloque, flags=re.S):
        pl = re.search(r'href="/[^"]*/profil/spieler/(\d+)"[^>]*>([^<]*)</a>', tr)
        cl = re.search(r'href="/[^"]*/startseite/verein/(\d+)"[^>]*>([^<]*)</a>\s*</td>', tr)
        tf = re.search(r'/transfer_id/(\d+)">(.*?)</a>', tr, flags=re.S)
        edad = re.search(r'<td class="zentriert">(\d+)</td>', tr)
        fee = None
        if tf:
            fee = re.sub(r"<[^>]+>", "|", tf.group(2)).split("|")[0].strip()
        out.append((
            pl.group(1) if pl else None,
            pl.group(2).strip() if pl else None,
            edad.group(1) if edad else None,
            cl.group(1) if cl else None,
            cl.group(2).strip() if cl else None,
            tf.group(1) if tf else None,
            fee,
        ))
    return out


def procesar(team_id, slug, y):
    c, page = get(f"{TM}/{slug}/transfers/verein/{team_id}/saison_id/{y}", fresco=(y >= DESDE))
    if str(c) == "404":
        return "sin_pagina", []
    if str(c) != "200":
        return f"http_{c}", []
    t = re.search(r"<title>([^<]*)</title>", page)
    titulo = t.group(1) if t else ""
    esperado = f"Transfers {str(y)[2:]}/{str(y + 1)[2:]}"
    if esperado not in titulo:
        return "titulo_distinto", []
    if not (re.search(r"<h2[^>]*>\s*Arrivals\s*</h2>", page) and re.search(r"<h2[^>]*>\s*Departures\s*</h2>", page)):
        return "sin_h2", []
    return "ok", filas(partir(page))


def levantar_api():
    try:
        urllib.request.urlopen(API + "/openapi.json", timeout=10)
        print("API ya levantada")
        return
    except Exception:
        pass
    url = f"https://github.com/felipeall/transfermarkt-api/archive/{API_REF}.zip"
    urllib.request.urlretrieve(url, "/tmp/tmapi.zip")
    z = zipfile.ZipFile("/tmp/tmapi.zip")
    raiz = "/tmp/" + z.namelist()[0].split("/")[0]
    z.extractall("/tmp")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", raiz + "[server]"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("pip fallo: " + r.stderr[-300:])
    env = dict(os.environ, PYTHONPATH=raiz)
    log = open("/tmp/uvicorn.log", "w")
    subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
                     cwd=raiz, env=env, stdout=log, stderr=log)
    for _ in range(30):
        time.sleep(2)
        try:
            urllib.request.urlopen(API + "/openapi.json", timeout=10)
            print("API levantada")
            return
        except Exception:
            pass
    raise RuntimeError("La API no levanto en 60 s")


def llamar(pid, ll):
    try:
        with urllib.request.urlopen(f"{API}/players/{pid}/{ll}", timeout=90) as x:
            return x.status, x.read().decode("utf-8", "replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", "replace")[:300]
    except urllib.error.URLError as e:
        return -1, str(e)[:300]
    except Exception as e:
        return -2, str(e)[:300]

## 2) Compuerta del ciclo y pausa por freno
- En modo prueba no hay compuerta.
- Si no, `motivo_no_corre("jugadores")` decide si corresponde correr. Si no corresponde, registra el motivo en `bronze_corridas` y termina con `SALTEADO: <motivo>`. Si corresponde, toma el inicio del ciclo vigente (`CICLO_INI`).
- Después consulta la última pausa registrada en `bronze_corridas`. Si todavía no venció, termina con `PAUSA hasta <fecha>`.

In [ ]:
# 2) Compuerta del ciclo y pausa por freno
if MODO_PRUEBA:
    CICLO_INI = INICIO   # sin ciclo: "este ciclo" = esta corrida (reintenta errores y refresca temporadas recientes)
    print("MODO PRUEBA: sin ciclo ni compuerta")
else:
    motivo_salida = motivo_no_corre("jugadores")
    if motivo_salida:
        registrar("saltea: " + motivo_salida)
        dbutils.notebook.exit("SALTEADO: " + motivo_salida)
    c_ciclo = ciclo_vigente()
    CICLO_INI = c_ciclo["inicio"]
    print(f"ciclo {c_ciclo['lunes']} | abierto {CICLO_INI}")

pausa_vigente = spark.sql(f"SELECT MAX(pausa_hasta) AS p FROM {CORR}").first().p
if pausa_vigente and pausa_vigente > ahora():
    registrar("pausa activa", pausa=pausa_vigente)
    dbutils.notebook.exit(f"PAUSA hasta {pausa_vigente}")

## 3) Fase A: entradas de los clubes, todas las temporadas y divisiones
Para cada club de `bronze_clubs` y cada temporada de `TEMPORADAS`, procesa la página de transfers y guarda solo las entradas en `bronze_pe_arrivals`. El resultado de cada página queda en `bronze_pe_status` (`estado`, cantidad de entradas y hora de bajada).

- **Qué se saltea:** los pares (club, temporada) con una temporada anterior a `DESDE` y sin estado `http_*` ni `error_*`, y los ya intentados en este ciclo (`bajado_en >= CICLO_INI`). El resto se baja, con la copia en caché renovada para las temporadas desde `DESDE`.
- **Escritura:** cada `LOTE` (50) páginas borra lo anterior de esos pares y escribe lo nuevo.
- **Cortes:** el tope de `MAX_HORAS` corta la fase. Un freno del sitio también la corta y registra una pausa de `PAUSA_HORAS` horas. En ambos casos se escribe lo pendiente del lote, se registra el motivo en `bronze_corridas` y el notebook termina con `RuntimeError` ("Fase A incompleta").

In [ ]:
# 3) Fase A: entradas de los clubes, todas las temporadas y divisiones
cols = [c.lower() for c in spark.table(T_CLUBS).columns]
if "club_id" not in cols:
    raise RuntimeError(f"bronze_clubs no tiene la columna club_id; columnas: {cols}")
col_slug = "slug" if "slug" in cols else None
clubs = {}
for r in spark.table(T_CLUBS).selectExpr("CAST(club_id AS STRING) AS club_id",
                                         (f"CAST({col_slug} AS STRING) AS slug" if col_slug else "'x' AS slug")).distinct().collect():
    clubs.setdefault(r.club_id, r.slug or "x")
clubs = sorted(clubs.items())

# Hecho = sin error en una temporada cerrada, o intentado en este ciclo (los errores se reintentan una vez por ciclo).
hechos = {(r.team_id, r.season) for r in spark.sql(f"""
    SELECT team_id, season FROM {T_PE_EST}
    WHERE (season < {DESDE} AND estado NOT LIKE 'http_%' AND estado NOT LIKE 'error_%')
       OR bajado_en >= '{CICLO_INI}'""").collect()}
pend_pag = [(t, s, y) for (t, s) in clubs for y in TEMPORADAS if (t, y) not in hechos]
if LIMITE_PAGINAS:
    pend_pag = pend_pag[:LIMITE_PAGINAS]
print(f"clubes: {len(clubs)} | temporadas {TEMPORADAS[0]}-{TEMPORADAS[-1]} | refresco desde {DESDE} | paginas a procesar: {len(pend_pag)}")

bufer_arr, bufer_est = [], []


def volcar_pe():
    global bufer_arr, bufer_est
    if bufer_est:
        cond = " OR ".join(f"(team_id = '{e[0]}' AND season = {e[1]})" for e in bufer_est)
        spark.sql(f"DELETE FROM {T_PE_ARR} WHERE {cond}")
        spark.sql(f"DELETE FROM {T_PE_EST} WHERE {cond}")
        if bufer_arr:
            spark.createDataFrame(bufer_arr, ESQ_LLEG).write.mode("append").saveAsTable(T_PE_ARR)
        spark.createDataFrame(bufer_est, ESQ_EST).write.mode("append").saveAsTable(T_PE_EST)
    bufer_arr, bufer_est = [], []


corte_a, pausa_a = None, None
t_a = time.time()
try:
    for i, (team_id, slug, y) in enumerate(pend_pag, 1):
        if time.time() - T0 > MAX_HORAS * 3600:
            raise Corte(f"tope de {MAX_HORAS} h alcanzado en las paginas")
        try:
            estado, fa = procesar(team_id, slug, y)
        except Freno as e:
            raise Corte(f"FRENO de Transfermarkt: {e}", fallo=True, pausa=True)
        except Exception as e:
            estado, fa = f"error_{type(e).__name__}", []
        bufer_arr += [(team_id, slug, y, f[0], f[1], f[2], f[3], f[4], f[5], f[6]) for f in fa]
        bufer_est.append((team_id, y, estado, len(fa), ahora()))
        if i % LOTE == 0:
            volcar_pe()
            el = time.time() - t_a
            print(f"{i}/{len(pend_pag)} | {el/60:.1f} min | restante aprox {(el/i)*(len(pend_pag)-i)/60:.0f} min", flush=True)
except Corte as c:
    corte_a = c
    if c.pausa:
        pausa_a = (datetime.now() + timedelta(hours=PAUSA_HORAS)).isoformat(timespec="seconds")
finally:
    volcar_pe()

if corte_a:
    registrar(str(corte_a), pausa=pausa_a)
    raise RuntimeError(f"Fase A incompleta: {corte_a}")
print("Fase A lista")

## 4) Fase B: candidatos -> llamada transfers -> decisión
**Conjuntos de clasificación**

| Conjunto | Contenido |
|---|---|
| `padres` | IDs de club de `bronze_clubs` y de `bronze_youth_teams_v2` |
| `juveniles` | IDs de equipo de `bronze_youth_teams_v2`, más los clubes de `bronze_clubes_externos` con rol `juvenil` cuyo club padre está en `padres` |
| `marcadores` | IDs fijos (Retired y tres clubes clasificados como `no_club`), más los clubes de `bronze_clubes_externos` con rol `no_club` |
| `NOMBRES_MARCADOR` | `without club`, `unknown`, `retired`, `career break`, `own youth`: un club con uno de esos nombres, o sin id, también es marcador |

**Funciones**

| Función | Qué hace |
|---|---|
| `fecha_ord(t)` | Fecha de un traspaso para ordenar y cortar: la fecha del traspaso; si no tiene, el 1 de julio del inicio de la temporada (`83/84` -> `1983-07-01`); si no tiene ninguna, `9999-99-99` ("no se sabe") |
| `primer_registro(ts)` | Toma el traspaso más viejo (sin los futuros). Su club de origen es el primer club; si es un marcador, el primer club es el destino |
| `en_alcance(t)` | `True` si la fecha del primer registro es `FECHA_CORTE` o posterior. Sin fecha ni temporada se deja pasar |
| `calcular_candidatos()` | Jugadores que aparecen en `bronze_pe_arrivals` y no están en el universo, ordenados por la temporada más vieja en que aparecen. Excluye a los que ya acumularon `MAX_INTENTOS` errores en este ciclo |
| `volcar_b()` | Escribe en este orden: `bronze_universo_extra`, `bronze_candidatos` y `bronze_player_raw`. Si la corrida se corta en el medio, el jugador queda como nuevo y `02_jugadores` lo baja solo |

**Decisión por jugador** (después de esperar `ESPERA_API` s y llamar a `transfers`)

| Resultado de la llamada | `decision` / `motivo` |
|---|---|
| HTTP 404 | `descartado` / `sin_datos_404` |
| Respuesta que no es JSON válido | `error` / `json_invalido` |
| Sin traspasos | `descartado` / `sin_traspasos` |
| Primer club en `padres` | `aceptado` / `primer_club_padre` |
| Primer club en `juveniles` | `aceptado` / `primer_club_juvenil_de_padre` |
| Otro primer club | `descartado` / `primer_club_otro` |
| Aceptado con primer registro anterior a `FECHA_CORTE` | `descartado` / `primer_registro_pre_corte` |
| Otro código HTTP | `error` / `http_<código>` |

Los aceptados se agregan a `bronze_universo_extra` y su respuesta se guarda en `bronze_player_raw` con la llamada `transfers`.

**Cortes:** tope de `MAX_HORAS`; HTTP `202`, `403` o `429` (con pausa de `PAUSA_HORAS` horas); API sin respuesta (`Connection refused`); `MAX_ERRORES_SEGUIDOS` errores seguidos. Se escribe lo pendiente, se registra la corrida en `bronze_corridas` y, si hubo un corte, el notebook termina con `RuntimeError` ("Fase B incompleta").

In [ ]:
# 4) Fase B: candidatos -> llamada transfers -> decision
# Conjuntos para clasificar el primer club
padres = {r.i for r in spark.sql(f"""
    SELECT CAST(club_id AS STRING) AS i FROM {T_CLUBS}
    UNION SELECT CAST(club_id AS STRING) FROM {T_YOUTH}""").collect()}
juveniles = {r.i for r in spark.sql(f"SELECT CAST(team_id AS STRING) AS i FROM {T_YOUTH}").collect()}
marcadores = {"123", "2113", "2077", "12604"}   # Retired y tres clubes clasificados como no_club
if spark.catalog.tableExists(T_EXT):
    for r in spark.sql(f"SELECT CAST(club_id AS STRING) AS i, rol, CAST(parent_club_id AS STRING) AS p FROM {T_EXT}").collect():
        if r.rol == "no_club":
            marcadores.add(r.i)
        elif r.rol == "juvenil" and r.p in padres:
            juveniles.add(r.i)
NOMBRES_MARCADOR = {"without club", "unknown", "retired", "career break", "own youth"}
print(f"padres: {len(padres)} | juveniles de padres: {len(juveniles)} | marcadores: {len(marcadores)}")


def es_marcador(c):
    return (not c.get("id")) or str(c.get("id")) in marcadores or (c.get("name") or "").strip().lower() in NOMBRES_MARCADOR


def fecha_ord(t):
    # Fecha para ordenar/cortar: la del traspaso; sin fecha, el 1 de julio del inicio de la temporada ("83/84" -> 1983-07-01).
    # Sin ninguna de las dos devuelve "9999-99-99" (se trata como "no se sabe").
    d = t.get("date")
    if d:
        return d[:10]
    m = re.match(r"(\d{2})/", str(t.get("season") or ""))
    if m:
        yy = int(m.group(1))
        return f"{1900 + yy if yy >= 50 else 2000 + yy}-07-01"
    return "9999-99-99"


def primer_registro(ts):
    # Traspaso mas viejo (sin los futuros). La API los da del mas nuevo al mas viejo: a igual fecha gana el de mayor indice.
    ts = [t for t in ts if not t.get("upcoming")]
    if not ts:
        return None, None
    orden = sorted(enumerate(ts), key=lambda x: (fecha_ord(x[1]), -x[0]))
    t = orden[0][1]
    cf, ct = t.get("clubFrom") or {}, t.get("clubTo") or {}
    club = cf if not es_marcador(cf) else ct
    return club, t


def en_alcance(t):
    # El primer registro tiene que caer dentro del alcance del proyecto (desde FECHA_CORTE).
    # Sin fecha ni temporada ("9999-99-99") se deja pasar.
    return fecha_ord(t) >= FECHA_CORTE


UNIVERSO_SQL = f"""
    SELECT CAST(player_id AS STRING) AS player_id FROM {T_ARR}
    UNION SELECT CAST(player_id AS STRING) FROM {T_DEP}
    UNION SELECT player_id FROM {T_EXTRA}
    UNION SELECT player_id FROM {T_CAND} WHERE decision IN ('aceptado', 'descartado')"""


def calcular_candidatos():
    errs = {r.player_id: r.n for r in spark.sql(f"""
        SELECT player_id, COUNT(*) AS n FROM {T_CAND}
        WHERE decision = 'error' AND fecha_decision >= '{CICLO_INI}' GROUP BY 1""").collect()}
    filas_ = spark.sql(f"""
        SELECT a.player_id, MIN(a.season) AS season_min
        FROM {T_PE_ARR} a LEFT ANTI JOIN ({UNIVERSO_SQL}) u ON a.player_id = u.player_id
        WHERE a.player_id IS NOT NULL
        GROUP BY a.player_id ORDER BY season_min, a.player_id""").collect()   # los mas viejos primero
    return [(r.player_id, r.season_min) for r in filas_ if errs.get(r.player_id, 0) < MAX_INTENTOS]


tareas = calcular_candidatos()
if LIMITE_JUGADORES:
    tareas = tareas[:LIMITE_JUGADORES]
print(f"candidatos a decidir: {len(tareas)} (~{len(tareas) * (ESPERA_API + 0.5) / 3600:.1f} h a {ESPERA_API} s de espera)")

b_cand, b_extra, b_raw = [], [], []


def volcar_b():
    # Orden: extras -> candidatos -> raw. Si se corta en el medio, el jugador queda como "nuevo" y 02_jugadores lo baja solo.
    if b_extra:
        spark.createDataFrame(b_extra, ESQ_EXTRA).write.mode("append").saveAsTable(T_EXTRA)
    if b_cand:
        spark.createDataFrame(b_cand, ESQ_CAND).write.mode("append").saveAsTable(T_CAND)
    if b_raw:
        spark.createDataFrame(b_raw, SCHEMA_RAW).write.mode("append").saveAsTable(RAW)
    b_cand.clear(); b_extra.clear(); b_raw.clear()


n, n_err, seguidos = 0, 0, 0
motivo_b, fallo_b, pausa_b = "TERMINADO", False, None
if tareas:
    levantar_api()
try:
    for pid, season_min in tareas:
        if time.time() - T0 > MAX_HORAS * 3600:
            raise Corte(f"tope de {MAX_HORAS} h alcanzado")
        time.sleep(ESPERA_API)
        code, body = llamar(pid, "transfers")
        n += 1
        if code in (202, 403, 429):
            raise Corte(f"FRENO: HTTP {code} en transfers del jugador {pid}", fallo=True, pausa=True)
        if code == -1 and "refused" in body.lower():
            raise Corte("La API no responde (Connection refused)", fallo=True)

        ahora_ = ahora()
        if code == 404:
            b_cand.append((pid, "descartado", "sin_datos_404", None, None, None, None, 0, ahora_))
            seguidos = 0
        elif code == 200:
            try:
                ts = json.loads(body).get("transfers") or []
            except Exception:
                ts = None
            if ts is None:
                n_err += 1; seguidos += 1
                b_cand.append((pid, "error", "json_invalido", None, None, None, None, None, ahora_))
            else:
                seguidos = 0
                club, t = primer_registro(ts)
                if t is None:
                    b_cand.append((pid, "descartado", "sin_traspasos", None, None, None, None, len(ts), ahora_))
                else:
                    cid, cnom = str(club.get("id") or ""), club.get("name")
                    edad, fecha = t.get("age"), (fecha_ord(t) if fecha_ord(t) != "9999-99-99" else None)
                    if cid in padres:
                        dec, mot = "aceptado", "primer_club_padre"
                    elif cid in juveniles:
                        dec, mot = "aceptado", "primer_club_juvenil_de_padre"
                    else:
                        dec, mot = "descartado", "primer_club_otro"
                    if dec == "aceptado" and not en_alcance(t):
                        dec, mot = "descartado", "primer_registro_pre_corte"
                    b_cand.append((pid, dec, mot, cid, cnom, edad, fecha, len(ts), ahora_))
                    if dec == "aceptado":
                        b_extra.append((pid, season_min, cid, mot, edad, fecha, ahora_))
                        b_raw.append((pid, "transfers", "hecho", 200, body, ahora_))
        else:
            n_err += 1; seguidos += 1
            b_cand.append((pid, "error", f"http_{code}", None, None, None, None, None, ahora_))

        if len(b_cand) >= VOLCAR_CADA:
            volcar_b()
        if n % 500 == 0:
            print(f"{n} llamadas | {(time.time()-T0)/3600:.2f} h", flush=True)
        if seguidos >= MAX_ERRORES_SEGUIDOS:
            raise Corte(f"{MAX_ERRORES_SEGUIDOS} errores seguidos (ultimo: http {code}, {body[:120]})", fallo=True)
except Corte as c:
    motivo_b, fallo_b = str(c), True
    if c.pausa:
        pausa_b = (datetime.now() + timedelta(hours=PAUSA_HORAS)).isoformat(timespec="seconds")
finally:
    volcar_b()

registrar(motivo_b, n, n_err, pausa_b)
print(f"{motivo_b} | llamadas: {n} | errores: {n_err} | {(time.time()-T0)/3600:.2f} h")
if fallo_b:
    raise RuntimeError("Fase B incompleta: " + motivo_b)

## 5) Resumen y cierre
Tres consultas de control:

1. Cantidad de jugadores por `decision` y `motivo` en `bronze_candidatos`.
2. Diagnóstico: clubes primeros de los aceptados como `primer_club_juvenil_de_padre`, con cantidad de jugadores y rango de fechas (hasta 50 filas).
3. Cantidad de páginas por `estado` en `bronze_pe_status`.

Salidas del notebook:

| Condición | Resultado |
|---|---|
| Se usó `LIMITE_PAGINAS` o `LIMITE_JUGADORES` | Termina con `PRUEBA con limites: no es una corrida completa` |
| Quedan candidatos sin decidir | `RuntimeError` |
| No queda nada pendiente | Termina con `OK \| universo extra: <cantidad>` |

In [ ]:
# 5) Resumen y cierre
display(spark.sql(f"""
    SELECT decision, motivo, COUNT(*) AS jugadores FROM {T_CAND} GROUP BY 1, 2 ORDER BY 1, 3 DESC"""))

# Diagnostico: aceptados cuyo primer club es un juvenil de uno de los clubes (el proceso actual no los habia captado).
# Mirar por que: equipo juvenil faltante en bronze_youth_teams_v2, temporada sin pagina, pagina sin_h2, etc.
display(spark.sql(f"""
    SELECT club_primero_id, club_primero_nombre, COUNT(*) AS jugadores,
           MIN(fecha_primer_registro) AS desde, MAX(fecha_primer_registro) AS hasta
    FROM {T_CAND} WHERE motivo = 'primer_club_juvenil_de_padre'
    GROUP BY 1, 2 ORDER BY jugadores DESC LIMIT 50"""))

display(spark.sql(f"SELECT estado, COUNT(*) AS paginas FROM {T_PE_EST} GROUP BY 1 ORDER BY 2 DESC"))

pendientes_final = calcular_candidatos()
print(f"candidatos pendientes: {len(pendientes_final)} | universo extra: {spark.table(T_EXTRA).count()}")
if LIMITE_PAGINAS or LIMITE_JUGADORES:
    dbutils.notebook.exit("PRUEBA con limites: no es una corrida completa")
if pendientes_final:
    raise RuntimeError(f"Quedan {len(pendientes_final)} candidatos sin decidir")
dbutils.notebook.exit("OK | universo extra: " + str(spark.table(T_EXTRA).count()))